# Mission 1: Data Efficiency

**Goal:** improve the base model by choosing *which* stories to fine-tune on. Your baseline metric is perplexity on the validation stories.

**Constraint:** choose at most 100 stories (`common.M1_MAX_EXAMPLES`) from the 10,000-story pool loaded in section 1, and train only on those.

**What you design:** a data selection algorithm, and how to train on the stories you select.

**What you submit:** your algorithm code, your trained model, and your team benchmark.

Edit only the cells marked **YOUR CODE**. Cells that start with `%%writefile` save your code into the `submission/` folder, which staff run when grading. After editing one of these cells, rerun it.

### 1. Setup

Do the setup steps each time you open the notebook:
1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. **Upload `common.py`.** Open the file browser on the left and upload it. Colab deletes uploaded files when the session ends, so upload it again each session.
3. **Run the cell below.**

`common.py` contains the shared code that staff also use for grading. Do not edit it.

In [ ]:
!pip install -q transformers datasets

import os
assert os.path.exists("common.py"), "Put common.py in the same folder as this notebook."
import common

common.set_seed()
os.makedirs("submission/benchmark_data", exist_ok=True)
print("Device:", common.get_device())

### 2. Load data and the base model

In [ ]:
pool, val = common.load_tinystories()

model, tokenizer = common.load_model()
print(f"Base model validation perplexity: {common.compute_perplexity(model, tokenizer, val):.2f}")

### 3. Your algorithm

`select_examples` returns the indices of the stories to train on. `train` fine-tunes the model on them. Staff run exactly these two functions, starting from the base model.

`common.finetune()` accepts `epochs`, `max_steps`, `lr`, and `batch_size`, or you can write your own training loop. Run `help(common.finetune)` to see all options.

In [ ]:
%%writefile submission/algorithm.py
import random
import common


def select_examples(pool: list[str], k: int) -> list[int]:
    """Return the indices of at most k distinct stories in `pool` to train on."""
    # YOUR CODE HERE (currently picks stories at random; replace with your method)
    return random.Random(common.SEED).sample(range(len(pool)), k)


def train(model, tokenizer, examples: list[str]):
    """Fine-tune `model` on `examples` and return it."""
    # YOUR CODE HERE (currently 3 epochs with default settings; change as you like)
    common.finetune(model, tokenizer, examples, epochs=3)
    return model

In [ ]:
select_examples = common.load_function("submission/algorithm.py", "select_examples")
train = common.load_function("submission/algorithm.py", "train")

idx = select_examples(pool, common.M1_MAX_EXAMPLES)
assert len(idx) <= common.M1_MAX_EXAMPLES, f"Selected {len(idx)} examples; the limit is {common.M1_MAX_EXAMPLES}."
assert len(set(idx)) == len(idx), "Indices must be distinct."
assert all(0 <= i < len(pool) for i in idx), "Indices must refer to stories in the pool."

model, tokenizer = common.load_model()
model = train(model, tokenizer, [pool[i] for i in idx])
ppl = common.compute_perplexity(model, tokenizer, val)
print(f"Your validation perplexity: {ppl:.2f}")

### 4. Your team benchmark 

Requirements:
- `team_benchmark(model, tokenizer)` returns **one number, where higher is better**
- it returns the same number every time it runs (seed any randomness)
- it finishes within `common.BENCHMARK_TIME_LIMIT_S` seconds and does not change the model's weights
- any data it needs goes in `submission/benchmark_data/`, and none of it may come from the training data loaded above

**Where to put your benchmark data.** Choose whichever is easier:
- **Few examples:** write them as a Python list directly in the cell below.
- **More examples, or data you generate with a script:** save a file (for example, `.json` or `.txt`) in `submission/benchmark_data/` and read it inside `team_benchmark`.

In [ ]:
%%writefile submission/benchmark.py
"""Team benchmark."""
import os
import common

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "benchmark_data")

# Option 1: a short list of examples written here
# EXAMPLES = [
#     'First example...',
#     'Second example...',
# ]

# Option 2: a file in submission/benchmark_data/
# import json
# with open(os.path.join(DATA_DIR, "my_data.json")) as f:
#     EXAMPLES = json.load(f)

def team_benchmark(model, tokenizer) -> float:
    """Evaluate `model` and return one number. Higher must mean better."""
    # YOUR CODE HERE
    raise NotImplementedError("Write your team benchmark.")

In [ ]:
team_benchmark = common.load_function("submission/benchmark.py", "team_benchmark")

base_model, base_tokenizer = common.load_model()
print("Base model:")
common.check_team_benchmark(team_benchmark, base_model, base_tokenizer)

print("Your model:", team_benchmark(model, tokenizer))

### 5. Save your submission

Run this when you are ready to submit. Then upload the zip file to Gradescope.

Your `submission/` folder should contain:
- `algorithm.py`
- `benchmark.py` and `benchmark_data/`
- `model/`

In [ ]:
common.save_model(model, tokenizer)
common.package_submission("mission1")